### Install New Libraries

In [ ]:
#!pip install ddgs trafilatura -q
#!pip install openai-agents

### Setup

In [ ]:
import os
from dotenv import load_dotenv
import json
from pprint import pprint
from IPython.display import Markdown, display
from ddgs import DDGS
import trafilatura

from agents import Agent, Runner, function_tool
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")

MODEL = "gpt-4.1-mini"

### Step 1: Define the Tools

In [ ]:
@function_tool
def search_web(query: str):
    """Search the web using DuckDuckGo browser. Returns 3 results."""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"  \u2705 search_web: Got Results for {query}\n")
    return json.dumps(results, indent=2)

In [ ]:
@function_tool
def fetch_url(url: str):
    """Fetch the URL content using Trafilatura and extract the text. Returns the extracted text if successful, otherwise a failure message."""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f"  \u2705 Got text: {len(text)} characters\n")
            return text
    print(f"  \u274C Failed to fetch or extract the text from {url}\n")
    return f"Could not fetch or extract text from {url}. Try a different source."

In [ ]:
import base64
from pathlib import Path
from openai import OpenAI

openai_client = OpenAI()

@function_tool
def generate_image(prompt:str) -> str:
    """Generate an image using GPT Image. The prompt should be a detailed visual description."""
    print(f"  🎨 generate_image: {prompt[:60]}...")
    response = openai_client.images.generate(
        model="gpt-image-1-mini",
        prompt=prompt,
        size="1536x1024",
        quality="high",
        n=1
    )

    base64_image = response.data[0].b64_json
    image_data = base64.b64decode(base64_image)
    notebook_dir = Path("AI Engineering Part 2")
    if not notebook_dir.exists():
        notebook_dir = Path.cwd()
    image_path = notebook_dir / "image_agent_output.png"
    with open(image_path, "wb") as f:
        f.write(image_data)

    print(f"  \u2705 generate_image: Image generated at {image_path}\n")
    return "image_agent_output.png"

### Step 2: The Agents

The Agent Prompts tell the LLM who it is and how to behave. The key things:

- What its job is
- What tools it has
- What process to follow
- What output format to produce

#### Research Agent

In [ ]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

***IMPORTANT:
After each search with search_web, you MUST first explain your reasoning:
- Which URLs look most relevant and why
- Which ones you will fetch and why
- Which ones you are skipping and why
Only AFTER writing out your reasoning should you call fetch_url.***

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

You MUST gather information from at least 3 distinct sources before delivering your brief.
If you have fewer than 3 sources, keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move.
"""

research_agent = Agent(
    name="Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model=MODEL,
    tools=[search_web, fetch_url]
)

### Image Generator Agent

In [ ]:
IMAGE_AGENT_PROMPT = """You are an image prompt specialist. Given a topic and content summary,
craft a detailed prompt GPT Image Prompt for a hero image.

Rules for your GPT Image Prompt:
- Describe a natural, photographic-style image (not illustrated, not cartoon)
- No text, logos, or words in the image
- No human faces or recognizable people
- No icon dumps or collages
- Focus on a single compelling visual that captures the theme
- Be specific about lighting, composition, and mood
- Keep the prompt under 200 words

Call generate_image exactly ONCE with your prompt. One image only.
After the tool returns, reply with only the returned image path and nothing else.

IMPORTANT: When returning the image URL, copy it EXACTLY character by character as returned by the image_agent tool. Do not add or remove any characters, spaces, or punctuation. Do not
modify, shorten or paraphrase the URL in any way. It must be an exact match to what the image_agent tool returned.
"""

image_agent = Agent(
    name="Image Agent",
    instructions=IMAGE_AGENT_PROMPT,
    model=MODEL,
    tools=[generate_image]
 )

image_agent_as_tool = image_agent.as_tool(
    tool_name="image_agent",
    tool_description="Generate a hero image for an article based on the topic and content summary. Supply the topic and content summary."
)

#### Orchestrator Agent

In [ ]:
ORCHESTRATOR_AGENT_PROMPT = RECOMMENDED_PROMPT_PREFIX + """
You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.

Once you have selected the best brief, you MUST use the image_agent tool to generate an image.
Use the research brief to suppply the image agent with the topic and content summary it needs to generate the image.

Then, decide which write is the best fit for this topic and research:
- The Journalist Agent: Writes in a professional, journalistic style, suitable for news articles and informative pieces.
- The Humorist Agent: Writes in a humorous, satirical style, suitable for entertainment and comedic articles.
- The Advisor Agent: Writes in an advisory, informative style, suitable for providing guidance and recommendations.

Then handoff to your chosen writer.

Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.
Do not write the article yourself or add anything, you MUST handoff to the appropriate writer agent.
"""

orchestrator_agent = Agent(
    name="Orchestrator Agent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model="o4-mini", ## reasoning for choosing this model is that it is lightweight and fast, suitable for orchestrating tasks without heavy computational overhead
    tools=[research_agent.as_tool(tool_name="research_agent", tool_description="A research specialist agent that gathers information and produces comprehensive research briefs.", max_turns=25),
           image_agent_as_tool]
)

#### Writer Agent A: The Journalist

In [ ]:
JOURNALIST_WRITER_PROMPT = RECOMMENDED_PROMPT_PREFIX + """ 
You are an investigative journalist. 
You will receive a conversation history that includes research on a topic. 

Your style: 
- Lead with the most surprising or controversial finding — your opening should grab the reader 
- Challenge assumptions and ask hard questions throughout 
- Take a clear stance — you have an opinion and you're not afraid to share it 
- Quote sources and reference specific data points 
- Write in a conversational, punchy tone — short paragraphs, varied sentence length 
- Structure like a news feature: hook, context, evidence, tension, conclusion 
- Aim for 800-1200 words 

Do NOT use generic section headers like "Introduction", "Overview" or "Conclusion." Instead, use descriptive subheadings that reflect the content of each section.
Do NOT overused headers. Only use one title and BETWEEN 2 and 3 headers in total for the entire article.
Do NOT use bullet points or numbered lists. Write in full sentences and paragraphs.
Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format. 

Include the hero image URL at the top of your article in markdown format: ![Hero Image](url)
IMPORTANT: Copy the image URL EXACTLY as provided. Do not modify it.
"""

journalist_agent = Agent(
    name="Journalist Agent",
    instructions=JOURNALIST_WRITER_PROMPT,
    model=MODEL
)

#### Writer Agent B: The Humorist

In [ ]:
HUMORIST_WRITER_PROMPT = RECOMMENDED_PROMPT_PREFIX + """
You are a humorist writer.
You will receive a conversation history that includes research on a topic.
Your task is to create humorous content based on the research provided.

Your style:
- Use wit, satire, and irony to entertain the reader
- Exaggerate and play with absurdity to highlight the humor in the topic
- Employ clever wordplay, puns, and unexpected twists to keep the reader engaged
- Maintain a lighthearted and playful tone throughout the article

Do NOT use generic section headers like "Introduction", "Overview" or "Conclusion." Instead, use descriptive subheadings that reflect the content of each section.
Do NOT overused headers. Only use one title and BETWEEN 2 and 3 headers in total for the entire article.
Do NOT use bullet points or numbered lists. Write in full sentences and paragraphs.
Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format.

Include the hero image URL at the top of your article in markdown format: ![Hero Image](url)
IMPORTANT: Copy the image URL EXACTLY as provided. Do not modify it.
"""

humorist_agent = Agent(
    name="Humorist Agent",
    instructions=HUMORIST_WRITER_PROMPT,
    model=MODEL
)

#### Writer Agent C: The Advisor Writer

In [ ]:
ADVISOR_WRITER_PROMPT = RECOMMENDED_PROMPT_PREFIX + """You are a strategic advisor writing a memo for decision-makers.
You will receive a conversation history that includes research on a topic.

Your style:
- Lead with why this matters to the reader right now — what's at stake
- Focus on "what does this mean for you" and "what should you do about it"
- Be direct and authoritative — every sentence should earn its place
- Break down complex findings into clear, specific recommendations
- End with concrete action items the reader can act on this week
- Write like you're advising a CEO, not lecturing a classroom
- Aim for 800-1200 words

Do NOT use generic section headers like "Introduction", "Conclusion", or "Overview". Use creative, specific headers that grab attention.
Do NOT overuse headers. Only use one title and BETWEEN 2 and 3 headers in total for the entire article.
Do NOT use bullet points or numbered lists.
Do NOT ask for feedback, offer revisions, or include any commentary after the article.
Just deliver the finished article in markdown format.

Include the hero image URL at the top of your article in markdown format: ![Hero Image](url)
IMPORTANT: Copy the image URL EXACTLY as provided. Do not modify it.
"""

advisor_agent = Agent(
    name="Advisor Agent",
    instructions=ADVISOR_WRITER_PROMPT,
    model=MODEL
)

#### Update the Orchestrator Agent

In [ ]:
# TEMPORARY
# MOdified orchestrator without tools -- for quick testing of the handoff process without running the research and image generation tools. This allows for faster iteration and debugging of the handoff logic between agents.
ORCHESTRATOR_AGENT_PROMPT = RECOMMENDED_PROMPT_PREFIX + """
You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Ignore the tools available to you.

As soon as you have the topic, right away decide which write is the best fit for this topic:
- The Journalist Agent: Writes in a professional, journalistic style, suitable for news articles and informative pieces.
- The Humorist Agent: Writes in a humorous, satirical style, suitable for entertainment and comedic articles.
- The Advisor Agent: Writes in an advisory, informative style, suitable for providing guidance and recommendations.

Then handoff to your chosen writer.

Do not write the article yourself or add anything, you MUST handoff to the appropriate writer agent.
Do not use tools.
"""

orchestrator_agent = Agent(
    name="Orchestrator Agent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model="o4-mini" ## reasoning for choosing this model is that it is lightweight and fast, suitable for orchestrating tasks without heavy computational overhead
)

In [ ]:
from agents import handoff
from pydantic import BaseModel

class WriterSelectionInfo(BaseModel):
    which_writer: str
    reason: str

async def on_handoff(_ctx, decision: WriterSelectionInfo):
    print(f" ✍️ Writer selected: {decision.which_writer} - Reason: {decision.reason}\n")

orchestrator_agent.handoffs = [
    handoff(agent=journalist_agent, on_handoff=on_handoff, input_type=WriterSelectionInfo), 
    handoff(agent=humorist_agent, on_handoff=on_handoff, input_type=WriterSelectionInfo), 
    handoff(agent=advisor_agent, on_handoff=on_handoff, input_type=WriterSelectionInfo)
]

### Let's Run It!

In [ ]:
from agents import trace

with trace("Article Writer w/ Handoff", group_id="Learning AI Engineering"):
    result = await Runner.run(
        orchestrator_agent,
        # [For later when we go back to the full system] input = "Research the following topic and produce a comprehensive research brief: The most recent earthquake in Venezuela and its impact on the local communities.",
        input = "Write an article about: The rise of AI Agents in the Workplace in the FMCG industry.",
        max_turns=30
    )

In [ ]:
print(f"Agent: {result.last_agent.name}")
print(f"----------------")
display(Markdown(result.final_output))